# Kazakh profanity detection — training, evaluation and inference

Runs in a fresh **Kaggle** notebook (Settings → Accelerator **GPU T4 x2 or P100**, Internet **On**) or in **Google Colab** (Runtime → GPU).
Fine-tunes `FacebookAI/xlm-roberta-base` on the prepared splits, selects the epoch and decision threshold on the validation set, evaluates **once** on the test set and saves the model.

Expected time on a T4: ~20–25 min. Set `REPO_URL` below to your GitHub repository.

In [ ]:
REPO_URL = "https://github.com/alibekramazanov/kazakh-profanity-detection.git"
HF_REPO_ID = ""   # optional: "<hf-user>/kazakh-profanity-xlmr", needs an HF_TOKEN secret (write access)

import os, subprocess
if not os.path.exists("kazakh-profanity-detection"):
    subprocess.run(["git", "clone", REPO_URL, "kazakh-profanity-detection"], check=True)
os.chdir("kazakh-profanity-detection")
print(os.getcwd())

In [ ]:
# Pinned versions (the GPU build of torch that comes with Kaggle/Colab is used)
!pip install -q -r requirements-train.txt
!nvidia-smi -L

## 1 · Data
The processed splits are in `data/processed/` (built from KazSAnDRA by `src/prepare_data.py`, seed 42).
The optional cell below rebuilds them from the original KazSAnDRA files and checks that they are identical.

In [ ]:
REBUILD_DATA = False
if REBUILD_DATA:
    !bash scripts/get_kazsandra.sh data/raw
    !python -m src.prepare_data --kazsandra_dir data/raw --out_dir data/rebuilt --seed 42
    import hashlib
    for s in ["train", "val", "test"]:
        h = [hashlib.md5(open(f"{d}/{s}.csv", "rb").read()).hexdigest() for d in ["data/processed", "data/rebuilt"]]
        print(s, "identical" if h[0] == h[1] else "DIFFERENT", h)

In [ ]:
import pandas as pd, json
for s in ["train", "val", "test"]:
    d = pd.read_csv(f"data/processed/{s}.csv")
    print(f"{s:5s} n={len(d):6d}  profanity={d.label.sum():5d}  clean={(d.label==0).sum():6d}  synthetic={d.is_synthetic.sum()}")
print(json.load(open("data/processed/dataset_stats.json"))["class_distribution_real"])

## 2 · Train (model and threshold selected on validation)

In [ ]:
!python -m src.train --config configs/train_config.json --data_dir data/processed --out_dir outputs

In [ ]:
h = json.load(open("outputs/results/train_history.json"))
pd.DataFrame(h["history"])

## 3 · Evaluate once on the held-out test set

In [ ]:
!python -m src.evaluate --model_dir outputs/model --data_dir data/processed --out_dir outputs/results

In [ ]:
from IPython.display import Image, display
m = json.load(open("outputs/results/test_metrics.json"))
print(json.dumps({k: m[k] for k in ["n_examples", "threshold", "profanity_class", "confusion_matrix", "pr_auc", "subgroups"]}, indent=2, ensure_ascii=False))
display(Image("outputs/results/test_confusion_matrix.png"))

In [ ]:
pd.set_option("display.max_colwidth", 200)
print("FALSE POSITIVES (clean, predicted profanity)")
display(pd.read_csv("outputs/results/test_false_positives.csv").head(15))
print("FALSE NEGATIVES (profanity, predicted clean)")
display(pd.read_csv("outputs/results/test_false_negatives.csv").head(15))

### Extra check: hand-written product-review challenge set
KazSAnDRA's marketplace reviews contain no profanity (they are moderated), so the test set cannot measure recall on product reviews. `data/challenge/challenge_set.csv` has 48 hand-written marketplace-style comments (24 profane, 24 clean hard negatives). It is synthetic and is NOT used for training or for choosing any setting.

In [ ]:
!python -m src.evaluate --model_dir outputs/model --data_dir data/challenge --split challenge_set --out_dir outputs/results
ch = pd.read_csv("outputs/results/challenge_set_predictions.csv").merge(pd.read_csv("data/challenge/challenge_set.csv")[["id", "note"]], on="id")
display(ch[ch.label != ch.pred][["text", "label", "prob", "note"]])

## 4 · Try your own comment (no retraining)

In [ ]:
from src.predict import ProfanityDetector
det = ProfanityDetector("outputs/model")
examples = ["Өте жақсы тауар, жылдам жеткізді, рахмет!",
            "Сапасы нашар екен, ақшамды қайтарыңдар",
            "қ*тақбас приложение, ашылмайды",
            "Тапсырыс келмеді, пиздец"]
for r in det.predict(examples):
    print(r["label"], f"{r['prob_profanity']:.3f}", r["text"])

In [ ]:
my_text = "Мұнда өз пікіріңізді жазыңыз"   # <- edit and run
det.predict(my_text)

## 5 · Save the model outside GitHub
Option A — push to the Hugging Face Hub (add a Kaggle secret `HF_TOKEN` with write access and set `HF_REPO_ID`).
Option B — download `model.zip` and `results.zip` from the Output panel; attach `model.zip` to a GitHub Release and commit `outputs/results/` to the repository.

In [ ]:
if HF_REPO_ID:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        token = os.environ.get("HF_TOKEN")
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(HF_REPO_ID, exist_ok=True)
    api.upload_folder(folder_path="outputs/model", repo_id=HF_REPO_ID)
    print("uploaded: https://huggingface.co/" + HF_REPO_ID)

import shutil
out = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
shutil.make_archive(f"{out}/model", "zip", "outputs/model")
shutil.make_archive(f"{out}/results", "zip", "outputs/results")
print(os.listdir(out))